In [1]:
from dotenv import load_dotenv
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

d:\PROGRAMMING\Agentic AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

@tool
def get_product(name: str) -> str:
    """Look up a product by name and return its price, rating, stock, and description."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)

In [3]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent = create_agent(
    llm,
    tools=[get_product],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

In [4]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [5]:
ask("what is the price of wireless headphones.")

The price of the wireless headphones is $79.99.


In [6]:
ask("tell me about mechanical keyboard")

The mechanical keyboard is a high-quality product with a price of $129. It has a rating of 4.8 out of 5 stars, indicating excellent customer satisfaction. The keyboard features Cherry MX Brown switches, which are known for their tactile and quiet typing experience. Additionally, it has per-key RGB lighting, allowing for customizable backlighting effects. The tenkeyless design makes it more compact and ideal for gamers or typists who prefer a more minimalist setup.


In [7]:
REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

@tool
def get_review(name: str) -> str:
    """Look up a product review by a product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not available for this product"
    return str(r)

In [17]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

def ask2(question: str):
    result = agent2.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [18]:
ask2("how do people like smart watch")

People generally like smartwatches, with an average rating of 4.3 out of 5 stars and over 340 reviews. The smartwatch is priced at $199.99 and has features such as tracking heart rate and sleep, a 5-day battery life, and is water-resistant.


In [25]:
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatGroq(model="llama-3.3-70b-versatile" , temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt= "you are a helpful product assistant for an online tech store",
    checkpointer= InMemorySaver()
)

def ask2(question: str):
    config = {"configurable" : {"thread_id" :  "user-alice-session-1"}}
    result = agent2.invoke(
        {"messages" :  [{"role" : "user", "content": question}]},
        config
    )
    print(result["messages"][-1].content)

In [26]:
ask2("what is the price of wireless headphones?")

The price of the wireless headphones is $79.99. They have a rating of 4.6 and a description of "Over-ear Bluetooth, 30-hr battery, active noise cancellation."


In [27]:
ask2("what's its ratings? ")

The wireless headphones have a rating of 4.6 based on 1262 reviews.
